In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

### Build yield curve sub model

In [2]:
bm_list_yc = []
# create yield curve build method
content_sofr = {
    "TARGET": "SOFR-1B",
    "OVERNIGHT INDEX FUTURE": "SOFR-FUTURE-3M",
    "OVERNIGHT INDEX SWAP": "USD-SOFR-OIS",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_sofr))
# funding build method
content_sofr_funding = {
    "TARGET": "SOFR-1B-FLAT",
    "SPREAD ZERO RATE": "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_sofr_funding))
# create yield curve common build method
content = {"TARGET": "USD", "FUNDING PARAMETERS": "USD-FUNDING-PARAMETERS", "SOLVER": "BRENTQ"}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_COMMON", content))
build_method_collection_yc = qfCreateModelBuildMethodCollection(bm_list_yc)

In [3]:
### ois futures
df_fut = pd.DataFrame(
    [
        ["2026-03-19x2026-06-18", 96.44],
        ["2026-06-18x2026-09-17", 96.70],
        ["2026-09-17x2026-12-10", 96.85],
        ["2026-12-10x2027-03-17", 96.90],
        ["2027-03-17x2027-06-16", 96.91],
        ["2027-06-16x2027-09-15", 96.89],
        ["2027-09-15x2027-12-15", 96.85],
        ["2027-12-15x2028-03-15", 96.81],
        ["2028-03-15x2028-06-21", 96.76],
        ["2028-06-21x2028-09-20", 96.71],
        ["2028-09-20x2028-12-20", 96.66],
        ["2028-12-20x2029-03-21", 96.61],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_fut = qfCreateData1D("OVERNIGHT INDEX FUTURE", "SOFR-FUTURE-3M", df_fut)

In [4]:
### ois swap
df_swap = pd.DataFrame(
    [
        ["4Y", 0.03358],
        ["5Y", 0.03422],
        ["6Y", 0.03491],
        ["7Y", 0.03560],
        ["8Y", 0.03624],
        ["9Y", 0.03685],
        ["10Y", 0.03742],
        ["12Y", 0.03849],
        ["15Y", 0.03974],
        ["20Y", 0.04087],
        ["25Y", 0.04110],
        ["30Y", 0.04089],
        ["35Y", 0.04044],
        ["40Y", 0.03996],
        ["50Y", 0.03887],
        ["60Y", 0.03772],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_swap = qfCreateData1D("OVERNIGHT INDEX SWAP", "USD-SOFR-OIS", df_swap)

In [5]:
# spread zero rate
df_spread_zero_rate_rfr = pd.DataFrame(
    [["1Y", 0.0], ["5Y", 0.0], ["10Y", 0.0], ["20Y", 0.0], ["30Y", 0.0]],
    columns=["axis1", "values"],
).set_index("axis1")
data_szr_rfr = qfCreateData1D(
    "SPREAD ZERO RATE", "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD", df_spread_zero_rate_rfr
)

In [6]:
### funding parameter table
df_dg = pd.DataFrame(
    [
        ["Overnight Index Future", "SOFR-FUTURE-3M", "SOFR-1B-FLAT"],
        ["Overnight Index Swap", "USD-SOFR-OIS", "SOFR-1B-FLAT"],
    ],
    columns=["DATA TYPE", "DATA CONVENTION", "FUNDING IDENTIFIER"],
)
data_fpt = qfCreateDataGeneric("DATA GENERIC", "USD-FUNDING-PARAMETERS", df_dg)

In [7]:
### pack up all data into data collection
data_collection_yc_list = [data_szr_rfr, data_fut, data_swap, data_fpt]
data_collection_yc = qfCreateDataCollection(data_collection_yc_list)

In [8]:
value_date = "2026-02-11"
yc_model = qfCreateModel(value_date, "YIELD_CURVE", data_collection_yc, build_method_collection_yc)
path = "serialized/yc_model_calibrated.pickle"
qfWriteModelObjectToFile(yc_model, path)

### Create Build Method Collection

In [9]:
build_method_type = 'IR_SABR'
content = {
    'TARGET' : 'SOFR-1B-SWAPTION',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR BETA' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR NU' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR RHO' : 'USD-SOFR-SWAPTION',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method = qfCreateBuildMethod(build_method_type, content)
display(sabr_build_method.display())

In [10]:
### serialize / de-serialize
path = 'sabr_build_method.pickle'
qfWriteBuildMethodToFile(sabr_build_method, path)
display(sabr_build_method.display())
sabr_build_method_back = qfReadBuildMethodFromFile(path)
# check
display(sabr_build_method_back.display())
# house keeping
os.remove(path)

In [11]:
### caplet
build_method_type = 'IR_SABR'
content_cf = {
    'TARGET' : 'SOFR-1B-CAPFLOOR',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR BETA' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR NU' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR RHO' : 'USD-SOFR-CAPFLOOR',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method_cf = qfCreateBuildMethod(build_method_type, content_cf)
display(sabr_build_method_cf.display())

In [12]:
### pack up as build method collection
bm_list = [sabr_build_method, sabr_build_method_cf]
bm_collection = qfCreateModelBuildMethodCollection(bm_list)
bm_collection.display()

In [13]:
bm_list_merged = bm_list + bm_list_yc
bm_collection_merged = qfCreateModelBuildMethodCollection(bm_list_merged)
bm_collection_merged.display()

### Data

In [14]:
### swaption
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1Y', '5Y', '10Y']
data_conv = 'USD-SOFR-SWAPTION'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [15]:
### capfloor
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1Y', '5Y', '10Y']
data_conv = 'USD-SOFR-CAPFLOOR'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv_cf = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta_cf = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu_cf= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho_cf = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [16]:
### create a data collection
data_list = [
    data2D_nv, data2D_beta, data2D_nu, data2D_rho,
    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection = qfCreateDataCollection(data_list)
display(data_collection.display())

In [17]:
data_collection_merged_list = data_list + data_collection_yc_list
data_collection_merged = qfCreateDataCollection(data_collection_merged_list)
display(data_collection_merged .display())

### create sabr model and pricing

In [18]:
sabr_model = qfCreateSABRModel(
    sub_model=yc_model,
    data_collection=data_collection,
    build_method_collection=bm_collection
)
print("SABR components:", sabr_model.components_.keys())

In [19]:
sabr_model = qfCreateModel(
    value_date,
    'IR_SABR',
    data_collection=data_collection_merged,
    build_method_collection=bm_collection_merged
)

In [20]:
sabr_model = SABRModelBuilder.create_sabr_model(
    sub_model=yc_model,
    data_collection=data_collection,
    build_method_collection=bm_collection
)

print("SABR components:", sabr_model.components_.keys())

In [21]:
params = sabr_model.get_sabr_parameters(
    IndexRegistry().get("SOFR-1B-CAPFLOOR"),
    0.5, 
    0.25 
)

print("interpolated SABR parameters:")
for k, v in params.items():
    print(k, v)

In [22]:
caplet = qfCreateProductRFRCapletFloorlet(
    effective_date="2026-06-17",
    expiry_offset="0D",
    term_or_termination_date="3M",
    payment_date="2026-09-21",
    on_index="SOFR-1B",
    strike=0.045,
    cap_or_floor="CAP",
    notional=1_000_000,
    accrual_basis="ACT/360",
    long_or_short="LONG",
)

qfDisplayProduct(caplet)

In [23]:
vp_funding = FundingIndexParameter({"Funding Index": "SOFR-1B-FLAT"})
vpc = ValuationParametersCollection([vp_funding])

request = ValuationRequest.PV

In [24]:
ve = ValuationEngineRFRCapletFloorlet(
    sabr_model,
    vpc,
    caplet,
    request,
)

ve.calculate_value()

print("PV           :", ve.value_)
print("Cash         :", ve.cash_)
print("DF           :", ve.df_)
print("Forward      :", ve.forward_)
print("Option Value :", ve.option_value_)
print("Expiry Date  :", ve.expiry_date_)
print("Tenor        :", ve.tenor_)

In [25]:
grad = []
ve.calculate_first_order_risk(grad)

for i, g in enumerate(grad):
    print(f"component {i}: shape={g.shape}")
    print(g)

In [26]:
cf_report = ve.create_cash_flows_report()
pv_cash_report = ve.get_value_and_cash()

display(cf_report.display())
display(pv_cash_report.display())

In [27]:
### test risk
df_risk = qfCreateValueReport(sabr_model, caplet, vpc, 'firstorderrisk').display()
df_risk

In [28]:
df_risk[np.abs(df_risk["VALUES"].astype(float)) > 1e-10]

In [29]:
print("TTE:", ve.time_to_expiry_)
print("Forward:", ve.forward_)
print("Strike:", ve.strike_)
print("NV/BETA/NU/RHO:", ve.sabr_result_.get(SABRParameters.NV), ve.sabr_result_.get(SABRParameters.BETA), ve.sabr_result_.get(SABRParameters.NU), ve.sabr_result_.get(SABRParameters.RHO))
print("Option Value:", ve.option_value_)

### test pricing swaption

In [30]:
swaption = qfCreateProductRFRSwaption(
    expiry_date="2026-06-17",
    effective_date="2026-06-17",
    term_or_termination_date="5Y",
    payment_off_set="2D",
    on_index="SOFR-1B",
    strike=0.04,
    pay_or_rec="PAY",
    notional=1_000_000,
    accrual_period="1Y",
    accrual_basis="ACT/360",
    floating_leg_accrual_period="3M",
    pay_business_day_convention="F",
    pay_holiday_convention="USGS",
    spread=0.0,
    compounding_method="COMPOUND",
    long_or_short="LONG",
)
qfDisplayProduct(swaption)

In [31]:
ve_swaption = ValuationEngineRFRSwaption(
    sabr_model,
    vpc,
    swaption,
    request,
)

ve_swaption.calculate_value()

print("PV              :", ve_swaption.value_)
print("Cash            :", ve_swaption.cash_)
print("Swap PV         :", ve_swaption.swap_value_)
print("Forward SwapRate:", ve_swaption.forward_swap_rate_)
print("Annuity         :", ve_swaption.annuity_)
print("Option Value    :", ve_swaption.option_value_)
print("Expiry Date     :", ve_swaption.expiry_date_)
print("Tenor           :", ve_swaption.tenor_)

In [32]:
grad = []
ve_swaption.calculate_first_order_risk(grad)

for i, g in enumerate(grad):
    print(f"component {i}: shape={g.shape}")
    print(g)

In [33]:
cf_report_swaption = ve_swaption.create_cash_flows_report()
pv_cash_report_swaption = ve_swaption.get_value_and_cash()

display(cf_report_swaption.display())
display(pv_cash_report_swaption.display())

In [34]:
### test risk
df_risk_swaption = qfCreateValueReport(sabr_model, swaption, vpc, 'firstorderrisk').display()
df_risk_swaption[np.abs(df_risk_swaption["VALUES"].astype(float)) > 1e-5]

### Bump Reval

In [35]:
pv_base = qfCreateValueReport(sabr_model, caplet, vpc, "pv")[0][1]
print(f"Base PV: {pv_base}")

### inspect non-zero SABR risks
df_risk_nonzero = df_risk_swaption[np.abs(df_risk["VALUES"].astype(float)) > 1e-10].copy()
display(df_risk_nonzero)

In [36]:
risk_data_type = "SWAPTION NORMAL VOLATILITY"
risk_data_convention = "USD-SOFR-CAPFLOOR"
risk_expiry = "3M"
risk_tenor = "1Y"

bump_size = 1e-4

In [37]:
data2D_nv_cf.display()

In [38]:
# Step 1: build bumped up model
df_nv_cf_bumped = data2D_nv_cf.display().copy()
cur_nv = df_nv_cf_bumped.loc[risk_expiry, risk_tenor]
df_nv_cf_bumped.loc[risk_expiry, risk_tenor] = cur_nv + bump_size
data2D_nv_cf_bumped = qfCreateData2D(risk_data_type, risk_data_convention, df_nv_cf_bumped)
data_list_bumped = [data2D_nv, data2D_beta, data2D_nu, data2D_rho,
                    data2D_nv_cf_bumped, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_bumped = qfCreateDataCollection(data_list_bumped)
sabr_model_bumped = qfCreateSABRModel(sub_model=yc_model,
                                      data_collection= data_collection_bumped,
                                      build_method_collection=bm_collection)

# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(sabr_model_bumped, caplet, vpc, 'pv')[0][1]

# Step 3 : bump-reval risk
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.10f}.')

# Step 4: analytic risk
analytic_risk = df_risk_swaption[
    (df_risk_swaption["DATA_TYPE"] == risk_data_type)
    & (df_risk_swaption["DATA_CONVENTION"] == risk_data_convention)
    & (df_risk_swaption["AXIS1"] == risk_expiry)
    & (df_risk_swaption["AXIS2"] == risk_tenor)
]['VALUES'].values[0]
analytic_scaled = analytic_risk * bump_size
print(f'Analytic risk is {analytic_scaled}')
print(f"Difference is {risk_bump_reval - analytic_scaled:.10e}.")

### Bump Reval for swaption

In [39]:
pv_base = qfCreateValueReport(sabr_model, swaption, vpc, "pv")[0][1]
print(f"Base PV: {pv_base}")

df_risk_swaption = qfCreateValueReport(sabr_model, swaption, vpc, 'firstorderrisk').display()
df_risk_swaption[np.abs(df_risk_swaption["VALUES"].astype(float)) > 1e-5]

In [40]:
risk_data_type = "SWAPTION NORMAL VOLATILITY"
risk_data_convention = "USD-SOFR-SWAPTION"
risk_expiry = "3M"
risk_tenor = "5Y"

bump_size = 1e-5

In [41]:
# Step 1: build bumped up model
df_nv_bumped = data2D_nv.display().copy()
cur_nv = df_nv_bumped.loc[risk_expiry, risk_tenor]
df_nv_bumped.loc[risk_expiry, risk_tenor] = cur_nv + bump_size
data2D_nv_bumped = qfCreateData2D(risk_data_type, risk_data_convention, df_nv_bumped)
data_list_bumped = [data2D_nv_bumped, data2D_beta, data2D_nu, data2D_rho,
                    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_bumped = qfCreateDataCollection(data_list_bumped)
sabr_model_bumped = qfCreateSABRModel(sub_model=yc_model,
                                      data_collection= data_collection_bumped,
                                      build_method_collection=bm_collection)

# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(sabr_model_bumped, swaption, vpc, 'pv')[0][1]

# Step 3 : bump-reval risk
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.10f}.')

# Step 4: analytic risk
analytic_risk = df_risk_swaption[
    (df_risk_swaption["DATA_TYPE"] == risk_data_type)
    & (df_risk_swaption["DATA_CONVENTION"] == risk_data_convention)
    & (df_risk_swaption["AXIS1"] == risk_expiry)
    & (df_risk_swaption["AXIS2"] == risk_tenor)
]['VALUES'].values[0]
analytic_scaled = analytic_risk * bump_size
print(f'Analytic risk is {analytic_scaled}')
print(f"Difference is {risk_bump_reval - analytic_scaled:.10e}.")